# Build the EC-Otyper database

Combines the established O-type loci with additional published loci into one GenBank database.

Output: `EC_Oag_NoNovel.gbk`

In [7]:
import pandas as pd
from Bio import SeqIO

## Established loci

The novel31 locus is relabelled O25b; O165 is dropped.

In [8]:
oag_genbank = []

for record in SeqIO.parse("Loci_Oag_Established/combined_established_otypes.gbff", "genbank"):
    o165 = False
    for feature in record.features:
                if feature.type == "source":
                    if feature.qualifiers["note"] == [f"O locus: OLnovel31", f"O type: Onovel31"]:
                        feature.qualifiers["note"] = [f"O locus: OL25b", f"O type: O25b"]
                if feature.type == "source":
                    if feature.qualifiers["note"] == [f"O locus: OL165", f"O type: O165"]:
                        o165 = True
    if not o165:
        oag_genbank.append(record)
len(oag_genbank)

164

## Additional published loci

Records in `Other-Oag-Loci/` are tagged with the O-type from their filename; missing source features are added.

In [9]:
from Bio.SeqFeature import SeqFeature, SimpleLocation
from pathlib import Path

# Specify the directory
folder_path = Path("Other-Oag-Loci/")
source = False
# Iterate over all items and filter for files
for file_path in folder_path.iterdir():
    if file_path.is_file():
        extension = file_path.name.split(".")[-1]
        serotype = file_path.stem.split("_")[1]
        serotype = serotype.split(".")[0]
        if extension == "gbk":
            for record in SeqIO.parse(file_path, "genbank"):
                for feature in record.features:
                    if feature.type == "source":
                        source = True
                        feature.qualifiers["note"] = [f"O locus: OL{serotype[1:]}", f"O type: {serotype}"]
                if source == False:
                    print(record.id)
                    feature_location = SimpleLocation(start=1, end=len(record.seq), strand=1)
                    record.features.append(SeqFeature(
    location=feature_location,
    type="source",                 # E.g., 'gene', 'CDS', 'misc_feature', 'promoter'
    qualifiers={"note": [f"O locus: OL{serotype[1:]}", f"O type: {serotype}"]},
))
                oag_genbank.append(record)
len(oag_genbank)

215

In [10]:
SeqIO.write(oag_genbank, "EC_Oag_NoNovel.gbk", "genbank")

215